In [ ]:
# 1. Dependencies Setup: Install required packages
print("Installing dependencies... This may take a few minutes.")

# Downgrade Numpy for compatibility and install training dependencies
!pip uninstall -y numpy
!pip install "numpy<2.0" ultralytics timm sahi grad-cam

import numpy as np
import ultralytics
print(f"Numpy Version: {np.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}")
print("\n[IMPORTANT] If Numpy is still 2.x, please click 'RESTART SESSION' in the Kaggle UI now!")

In [ ]:
# 2. Environment Setup & Source Hotfix
from pathlib import Path
import sys, os, shutil

KAGGLE_DATASET = Path("/kaggle/input/datasets/mohamedtamzirt1")
RAW_SOURCE = KAGGLE_DATASET / "weapon-detection-source/weapon_source"
WRITABLE_SOURCE = Path("/kaggle/working/weapon_source")

# --- HOTFIX: Copy code to /kaggle/working and apply Multi-GPU fixes ---
if not WRITABLE_SOURCE.exists():
    print("Applying source code hotfixes for Multi-GPU stability...")
    shutil.copytree(str(RAW_SOURCE), str(WRITABLE_SOURCE))
    
    # Overwrite with fixed YOLOBackbone (Optimized for Multi-GPU)
    with open(WRITABLE_SOURCE / "models/backbones/yolo_backbone.py", "w") as f:
        f.write('import torch\nimport torch.nn as nn\nfrom ultralytics import YOLO\n\nclass YOLOBackbone(nn.Module):\n    def __init__(self, model_variant="yolo11m.pt", pretrained=True, freeze_backbone_epochs=5, intermediate_layers=None):\n        super().__init__()\n        if intermediate_layers is None:\n            if "yolo11" in model_variant or "yolo12" in model_variant:\n                self.intermediate_layers = [4, 6, 10]\n            else:\n                self.intermediate_layers = [4, 6, 9]\n        else:\n            self.intermediate_layers = intermediate_layers\n        \n        base_model = YOLO(model_variant)\n        self.model = base_model.model\n        \n        # Optimization: Only track layers up to P5\n        last_target_idx = max(self.intermediate_layers)\n        self.save = set(self.intermediate_layers)\n        for i, m in enumerate(self.model.model):\n            if i > last_target_idx: break\n            if hasattr(m, \'f\'):\n                if isinstance(m.f, int):\n                    if m.f != -1: self.save.add(m.f)\n                else:\n                    for f in m.f:\n                        if f != -1: self.save.add(f)\n        \n        device = next(self.model.parameters()).device\n        dummy_input = torch.zeros(1, 3, 640, 640).to(device)\n        features = self.forward(dummy_input)\n        self.out_channels = {k: v.shape[1] for k, v in features.items()}\n\n    def forward(self, x):\n        y = []\n        features = {}\n        last_target_idx = max(self.intermediate_layers)\n        for i, m in enumerate(self.model.model):\n            if m.f != -1:\n                if isinstance(m.f, int):\n                    x = y[m.f]\n                else:\n                    x = [x if j == -1 else y[j] for j in m.f]\n            x = m(x)\n            y.append(x if i in self.save else None)\n            if i == self.intermediate_layers[0]: features["P3"] = x\n            elif i == self.intermediate_layers[1]: features["P4"] = x\n            elif i == self.intermediate_layers[2]: features["P5"] = x\n            if i == last_target_idx: break\n        return features\n\n    def freeze(self):\n        for param in self.model.parameters(): param.requires_grad = False\n    def unfreeze(self):\n        for param in self.model.parameters(): param.requires_grad = True\n')
        
    # Overwrite with fixed HybridModel (Includes safety checks)
    with open(WRITABLE_SOURCE / "models/hybrid_model.py", "w") as f:
        f.write('import torch\nimport torch.nn as nn\nfrom models.backbones.yolo_backbone import YOLOBackbone\nfrom models.necks.swin_neck import SwinNeck\nfrom models.heads.detection_head import DetectionHead\nimport numpy as np\n\nclass HybridWeaponDetector(nn.Module):\n    def __init__(self, backbone_variant="yolo11m.pt", pretrained=True, nc=3, device="cuda"):\n        super().__init__()\n        self.device = device\n        self.nc = nc\n        self.backbone = YOLOBackbone(model_variant=backbone_variant, pretrained=pretrained)\n        in_channels = self.backbone.out_channels\n        self.neck = SwinNeck(in_channels=in_channels, embed_dim=512, num_heads=8, window_size=7, num_blocks=2)\n        self.head = DetectionHead(in_channels=in_channels, nc=nc)\n        self.to(device)\n\n    def forward(self, x):\n        features = self.backbone(x)\n        for key in ["P3", "P4", "P5"]:\n            if key not in features: raise KeyError(f"Backbone missing {key}")\n        enriched_features = self.neck(features)\n        return self.head(enriched_features)\n\n    def predict(self, frame, conf_threshold=0.25):\n        self.eval()\n        with torch.no_grad():\n            if isinstance(frame, np.ndarray):\n                if frame.shape[:2] != (640, 640):\n                    import cv2\n                    frame = cv2.resize(frame, (640, 640))\n                x = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0\n                x = x.unsqueeze(0).to(self.device)\n            else:\n                x = frame.to(self.device)\n            cls_logits, bbox_offsets, objectness = self.forward(x)\n            probs = torch.sigmoid(cls_logits) * torch.sigmoid(objectness)\n            conf, class_ids = torch.max(probs, dim=2)\n            mask = conf > conf_threshold\n            detections = []\n            valid_conf = conf[0][mask[0]]; valid_ids = class_ids[0][mask[0]]\n            for c, cid in zip(valid_conf, valid_ids):\n                detections.append({"bbox": [0, 0, 50, 50], "class_id": int(cid), "class_name": ["Weapon", "Person", "Confuser"][int(cid)], "confidence": float(c)})\n            return detections\n\n    def save(self, path: str):\n        torch.save(self.state_dict(), path)\n\n    @classmethod\n    def load(cls, path: str, device: str = "cuda"):\n        model = cls(device=device)\n        model.load_state_dict(torch.load(path, map_location=device))\n        model.to(device); model.eval()\n        return model\n')
    print("✅ Source code hotfixed and ready.")

SOURCE_ROOT = WRITABLE_SOURCE
DATA_ROOT = KAGGLE_DATASET / "wd-data/yolo_dataset"
DATA_YAML = DATA_ROOT / "data.yaml"
WEIGHTS_DIR = Path("/kaggle/working/weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

if str(SOURCE_ROOT) not in sys.path:
    sys.path.append(str(SOURCE_ROOT))

os.chdir("/kaggle/working")
print(f"✅ CWD: {os.getcwd()}")
print(f"✅ Source imported from: {SOURCE_ROOT}")

In [ ]:
# 3. Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import yaml
from pathlib import Path

try:
    from models.hybrid_model import HybridWeaponDetector
    from ultralytics.data.dataset import YOLODataset
    from ultralytics.data.utils import check_det_dataset
    print("✅ Project modules imported successfully.")
except ImportError as e:
    print(f"[ERROR] Import failed: {e}")
    print("Check if 'weapon-detection-source' contains 'models/' and 'ultralytics' is installed.")

In [ ]:
def get_dataloaders(data_yaml_path, data_root, batch_size=32, imgsz=640):
    data_cfg = check_det_dataset(data_yaml_path)
    
    # --- Automatic Path Alignment ---
    for split in ['train', 'val', 'test']:
        if split in data_cfg:
            data_cfg[split] = str(data_root / split / "images")
    
    # CRITICAL: Disable all caching to save RAM. 
    # Caching 41k images/labels can hit the 30GB Kaggle limit instantly.
    train_set = YOLODataset(img_path=data_cfg['train'], imgsz=imgsz, augment=True, batch_size=batch_size, task='detect', data=data_cfg)
    val_set = YOLODataset(img_path=data_cfg['val'], imgsz=imgsz, augment=False, batch_size=batch_size, task='detect', data=data_cfg)
    
    # RAM OPTIMIZATION:
    # - Reduced num_workers to 2 (each worker uses a lot of RAM on Kaggle)
    # - pin_memory=False to avoid pre-allocating locked RAM
    # - persistent_workers=False to free memory between epochs
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=False, collate_fn=train_set.collate_fn)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=False, collate_fn=val_set.collate_fn)
    
    return train_loader, val_loader

### 3. HybridTrainer (Dual GPU + AMP + Class Weighting)

In [ ]:
class HybridTrainer:
    def __init__(self, model, train_loader, val_loader, device="cuda", weights_dir=None):
        self.device = device
        self.weights_dir = Path(weights_dir)
        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
            self.model = nn.DataParallel(model).to(device)
        else:
            self.model = model.to(device)
            
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.scaler = torch.cuda.amp.GradScaler()
        base_model = self.model.module if hasattr(self.model, 'module') else self.model
        base_model.head.alpha = torch.tensor([1.0, 1.0, 2.5], device=device)
        # Standard Criterion: Loss computed on the main GPU
        self.criterion = base_model.head.compute_loss
        self.best_val_loss = float('inf')

    def train_epoch(self, optimizer, epoch):
        self.model.train()
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
        total_loss = 0
        for batch in pbar:
            imgs = batch['img'].to(self.device).float() / 255.0
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                # Back to standard call: DataParallel splits images, but we compute loss on gathered preds
                preds = self.model(imgs)
                loss = self.criterion(preds, batch, self.device)
            
            self.scaler.scale(loss).backward()
            self.scaler.step(optimizer)
            self.scaler.update()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        return total_loss / len(self.train_loader)

    def validate(self):
        self.model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in self.val_loader:
                imgs = batch['img'].to(self.device).float() / 255.0
                with torch.cuda.amp.autocast():
                    preds = self.model(imgs)
                    loss = self.criterion(preds, batch, self.device)
                val_loss += loss.item()
        return val_loss / len(self.val_loader)

    def run(self, epochs=50):
        base_model = self.model.module if hasattr(self.model, 'module') else self.model
        
        resume_path = self.weights_dir / "last.pt"
        start_epoch = 0
        if resume_path.exists():
            print(f"Resuming from {resume_path}")
            checkpoint = torch.load(resume_path)
            base_model.load_state_dict(checkpoint['model_state_dict'])
            start_epoch = checkpoint['epoch']
        
        # Robust Unfreeze Check: Ensure backbone is unfrozen if starting at epoch >= 10
        if start_epoch >= 10:
            print("[INFO] Unfreezing backbone for fine-tuning (Resumed session)")
            base_model.backbone.unfreeze()
        else:
            base_model.backbone.freeze()

        # Re-initialize optimizer with current requires_grad status
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, self.model.parameters()), lr=1e-4 if start_epoch < 10 else 1e-5)
        if resume_path.exists():
            try:
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            except:
                print("[WARNING] Optimizer state mismatch, starting with fresh LR")

        for epoch in range(start_epoch + 1, epochs + 1):
            if epoch == 10:
                print("\n[INFO] Unfreezing backbone for fine-tuning...")
                base_model.backbone.unfreeze()
                # Update optimizer to include backbone parameters
                optimizer = optim.AdamW(self.model.parameters(), lr=1e-5)
            
            avg_loss = self.train_epoch(optimizer, epoch)
            val_loss = self.validate()
            print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f}")
            
            if epoch % 2 == 0:
                torch.save({'epoch': epoch, 'model_state_dict': base_model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, self.weights_dir / "last.pt")
            
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(base_model.state_dict(), self.weights_dir / "best.pt")
                print(f"--> Best model saved with val_loss: {val_loss:.4f}")

### 4. Production Initiation

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target Hardware: {device}")

# We stay with the original input paths to avoid memory-intensive file copying
BACKBONE_PATH = "/kaggle/working/yolo11m.pt"
model = HybridWeaponDetector(
    backbone_variant=BACKBONE_PATH, 
    pretrained=True, 
    nc=3, 
    device=device
)

if DATA_YAML.exists():
    # Batch size 2 is the most stable setting for 15GB T4 GPUs with a Hybrid Transformer
    train_loader, val_loader = get_dataloaders(DATA_YAML, DATA_ROOT, batch_size=2)
    trainer = HybridTrainer(model, train_loader, val_loader, device=device, weights_dir=WEIGHTS_DIR)
    trainer.run(epochs=50)
else:
    print(f"[ERROR] data.yaml not found. Checked: {DATA_YAML}")